## RNN Fundamentals

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding,Masking,LSTM, GRU, Conv1D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import reuters
from keras.preprocessing import sequence
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import huggingface_hub as hub

## Helper Functions

In [12]:
# function to compute the accuracy, precision, recall and F1 score of a model's predictions.
def calculate_results(y_true, y_pred):
    model_accuracy = accuracy_score(y_true, y_pred)
    model_precision, model_recall, model_f1,_ = precision_recall_fscore_support(y_true, y_pred,average="weighted")
    model_results = {"accuracy":model_accuracy,
                     "precision":model_precision,
                     "recall" :model_recall,
                     "f1":model_f1}
    return model_results

In [13]:
train_df = pd.read_csv('train.csv')
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head(10)

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0
5559,7934,rainstorm,NaN,@Calum5SOS you look like you got caught in a r...,0
1765,2538,collision,NaN,my favorite lady came to our volunteer meeting...,1
1817,2611,crashed,NaN,@brianroemmele UX fail of EMV - people want to...,1
6810,9756,tragedy,"Los Angeles, CA",Can't find my ariana grande shirt this is a f...,0
4398,6254,hijacking,"Athens,Greece",The Murderous Story Of AmericaÛªs First Hijac...,1


In [14]:
xtrain, xtest, ytrain, ytest = train_test_split(
    train_df_shuffled['text'], train_df_shuffled['target'], test_size=0.1, random_state=42
)

xtrain.shape, ytrain.shape

((6851,), (6851,))

In [15]:
xtrain[0:5]

5921    @mogacola @zamtriossu i screamed after hitting...
3904              Imagine getting flattened by Kurt Zouma
2804    @Gurmeetramrahim #MSGDoing111WelfareWorks Gree...
3718    @shakjn @C7 @Magnums im shaking in fear he's g...
1667    Somehow find you and I collide http://t.co/Ee8...
Name: text, dtype: str

## Text Vectorization

In [16]:
text_vectorizer = TextVectorization(
    max_tokens=None,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=None,
    output_mode='int',
    output_sequence_length=None
)

max_vocab = 10000
max_length = 15

embedding = Embedding(
    input_dim=max_vocab, output_dim=128, input_length=max_length
)

c:\Users\LEI\Documents\deep-learning-ibm\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:119: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Model

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(repo_id="Dimitre/universal-sentence-encoder")

# 2. Load the standard TensorFlow SavedModel from the directory
use_model = tf.saved_model.load(model_dir)

# 3. Build your custom Keras wrapper layer
class USEEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, trainable=False, **kwargs):
        super().__init__(**kwargs)
        self.use_model = use_model
        self.trainable = trainable

    def call(self, inputs):
        # Universal Sentence Encoder accepts string tensors and outputs 512-dim vectors
        return self.use_model(inputs)

# 4. Instantiate the layer
encoder_layer = USEEmbeddingLayer(name="pretrained")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [1]:
model = tf.keras.Sequential([
                             encoder_layer,
                             layers.Dense(1,activation="sigmoid")], name="model_pretrained")
model.compile(loss="binary_crossentropy",
                     optimizer="adam",
                     metrics=["accuracy"])

model.fit(xtrain,
              y=ytrain,
              epochs=20,
              validation_data=(xtest,ytest))

calculate_results(y_true=ytest,
                  y_pred=tf.squeeze(tf.round(model.predict(xtest))))

NameError: name 'tf' is not defined